In [ ]:
import polars as pl
from datetime import datetime

In [ ]:
# Open file, add metadata and write to parquet
filename = "results.csv"
with open('data/results.csv', mode='r') as file:
    check = pl.scan_csv(file, has_header = True, separator = ",")
    
df = check.with_columns(
    pl.lit((datetime.now().strftime("%Y-%m-%dT%H:%M:%S"))).alias("date_time_ingested"),
    pl.lit(("kaggle")).alias("source")
)

# Write to parquet 

df.sink_parquet("data/bronze_landing_data.parquet")

In [ ]:
#Load the parquet file

df = pl.read_parquet("data/bronze_landing_data.parquet")


In [ ]:
df = df.cast({"home_team": pl.String, 
			"away_team": pl.String, 
			"home_score": pl.Int64, 
			"away_score": pl.Int64,
			"competition": pl.String,
			"stadium": pl.String,
			"city": pl.String, 
            "country": pl.String,
            "neutral": pl.Boolean,
            "world_cup": pl.Boolean,
            "source": pl.String})

df = df.with_columns(
    pl.col("date").str.to_datetime(format="%Y-%m-%d"),
    pl.col("date_time_ingested").str.to_datetime(format="%Y-%m-%dT%H:%M:%S")
    )
df.sink_parquet("data/bronze_raw_data.parquet")

In [ ]:
# Load the bronze
df = pl.read_parquet("data/bronze_landing_data.parquet")

# Add the source key, keep the seed key the same
df = df.with_columns(
    pl.struct(["date", "home_team", "away_team"]).hash(seed=0).alias("row_hash")
)

# Clean the string columns
string_cols = ["home_team", "away_team", "competition", "stadium", "city", "country", ]

df = df.with_columns(
    pl.col(c).str.strip_chars().str.to_titlecase() for c in string_cols
)

# # Check for nulls & convert if needed.
# df.filter((pl.col("home_team").str.to_lowercase() == "null") | (pl.col("competition") == "Premier League"))

# check = ["home_team","away_team", "home_score", "away_score"]


In [ ]:
# Null values allowed
required_cols = ["date", "home_team", "away_team", "home_score", "away_score", "competition"] # including competition as it 

violations = df.filter(
    pl.any_horizontal([pl.col(c).is_null() for c in required_cols])
)

violation_expr = pl.any_horizontal([pl.col(c).is_null() for c in required_cols])

violations = df.filter(violation_expr)
clean = df.filter(~violation_expr)

# write to the quarintine files 

violations.write_parquet("data/silver_quarintine_kaggle.parquet")

# Null value logic for allowed nulls

In [ ]:
# Dedup, value standardization & key identification

de_dup = clean.unique(pl.col("row_hash"))

# Value Standerdizations: Country, Stadium, teams, Competition, 

silver_reference_team = pl.DataFrame({"team":["England", "Australia", "Wales", "South Africa", "France", "Ireland", "Scotland", "New Zealand", "Argentina","Italy"]})
silver_reference_country = pl.DataFrame({"country":["England", "Australia", "Wales", "South Africa", "France", "Ireland", "Scotland", "New Zealand", "Argentina","Italy"]})
# Competition needs to be thought through more as the include 
silver_reference_stadium = pl.DataFrame({"stadium": ["Stade De France","Stadium Australia","Rectory Field","Ballymore Stadium","Inverleith","Stadio Plebiscito","Estadio José Fierro","Stade Pierre-Mauroy","Perth Stadium","Hamilton Crescent","Estadio Ricardo Etcheverry","Stadio Olimpico","St. Helen'S","Jade Stadium","Nelson Mandela Bay Stadium","Stadio Luigi Ferraris","Whalley Range","Olympic Park Stadium","Estadio Gimnasia Y Esgrima De Buenos Aires","Adelaide Oval","Perth Oval","Estadio Gigante De Arroyito","Stade Municipal","Rotorua Int. Stadium","Cardigan Fields","Meanwood Road","North Queensland Stadium","Estadio Centenario","Estadio José Amalfitani","Estadio Raúl Conti","Stade Du Moulias","Kingsmead Cricket Ground","Estadio Brigadier General Estanislao López","Concord Oval","Aviva Stadium","Sydney Cricket Ground","Westpac Stadium","Lancaster Park","Balmoral Showgrounds","Wembley Stadium","Stadio Olimpico Di Torino","Loftus Versfeld","Stade Geoffroy-Guichard","West Of Scotland F.C.","International Stadium Yokohama","Suncorp Stadium","Stadio Comunale Mario Battaglini","Ballymore","Shizuoka Stadium Ecopa","Stadio Arturo Collana","Colonial Stadium","Sydney Sports Ground","Estadio Estanislao López","Parc Olympique","Brisbane Cricket Ground","Yves-Du-Manoir","Estadio B.G Estanislao López","Ami Stadium","Rugby League Park","Rfk Stadium","St George'S Park","Estadio San Juan Del Bicentenario","Westpactrust Stadium","St Helen'S","Athletic Park","Stade Marcel Michelin","Millennium Stadium","Canberra Stadium","Stadium Municipal","José Amalfitani Stadium","Kings Park Stadium","Ferro Carrill Oeste","Sydney Football Stadium","Kings Park","Optus Stadium","Soldier Field","Newlands","Colombes","Galpharm Stadium","Gare De La Croix Du Prince","Nissan Stadium","San Siro Stadium","Stade Armandie","Melbourne Cricket Ground","King'S Park","Grand Stade Lille Métropole","Stadio Renato Dall'Ara","St Helens","Welford Road Stadium","Powderhall Stadium","Parc Des Princes","Murrayfield Stadium","Lansdowne Road","Stade Yves-Du-Manoir","Ōita Stadium","Mcalpine Stadium","Melbourne Rec. Stadium","Rodney Parade","Estadio Etcheverry","Epru Stadium","Mclean Park","Estadio 23 De Agosto","Stadio Artemio Franchi","Stade De Gerland","Vodacom Park","Powderhall","Stade De La Mosson","Buffalo City Stadium","Estadio Mario Alberto Kempes","Regional Stadium","Trafalgar Park","Kingsmead","Estadio Madre De Ciudades","Ashton Gate","Fallowfield","Stade De La Meinau","Stade Mayol","Allianz Riviera","Stade Félix Bollaert","Royal Bafokeng Stadium","Richardson'S Field","Crystal Palace","Ellis Park","Carisbrook","Stade Vélodrome","Hong Kong Stadium","Stadio Flaminio","Stade Maurice Trélut","Ormeau Cricket Ground","Vélez Sársfield","Murrayfield","Free State Stadium","Ferrocarril Stadium","Stade De La Beaujoire","Mardyke","Ravenhill Stadium","Mount Smart Stadium","Stadio Comunale Di Monigo","Croke Park","St James Park","Exhibition Ground","Epsom Showgrounds","Sky Stadium","Stade Marcel Saupin","Stade Lesdiguières","Gigante De Arroyito","Mcdonald Jones Stadium","Fc Oeste","Cbus Super Stadium","Upper Park","Kingsholm","Johann Van Riebeeck Stadium","Twickenham","Cardiff Arms Park","Eden Park","Ulster Cricket Ground","Docklands Stadium","Parc Y Scarlets","Stade Colombes","Estadio Padre Ernesto Martearena","Melbourne Rectangular Stadium","Stade Chaban-Delmas","Stade Des Ponts Jumeaux","Waikato Stadium","Stadio Euganeo","Olympic Stadium","Forsyth Barr Stadium","José Amalfitani","Hampden Park","Stradey Park","Ferrocaril Oeste","Thomond Park","Estadio Único","River Plate Stadium","Welford Road","Estadio Bicentenario","Pam Brink Stadium","Stadium Nord Lille Métropole","Telstra Dome","Stadio Marc'Antonio Bentegodi","Stadio Marassi","Stadio Mompiano","Twickenham Stadium","Raeburn Place","Loftus Versfeld Stadium","Malvinas Argentinas","Estadio José María Minella","Robina Stadium","Ellis Park Stadium","Absa Stadium","Boet Erasmus Stadium","Parc Lescure","Subiaco Oval","Newlands Stadium","Otago Stadium","Brisbane Exhibition Ground","Stadium Lille-Metropole","Tokyo Stadium","Crusaders Ground","Headingley","North Harbour Stadium","Tahuna Park","Arena Civica","Crown Flatt","Stadio Comunale Beltrametti","Rathmines","National Stadium","Lang Park","Estadio Monumental José Fierro","Yarrow Stadium","Estadio Malvinas Argentinas","Wellington Regional Stadium","Athletic Ground","Bankwest Stadium","The Oval","Fnb Stadium","Mbombela Stadium","Ravenhill","Padre Ernesto Martearena","Cape Town Stadium","Estadio Olímpico","Birkenhead Park","Bruce Stadium","Rugby Park","Springbok Park","Telstra Stadium","Stadio Xxv Aprile","Stadio Friuli","Old Trafford","St James' Park","Stade Pershing", "Commbank Stadium"]})

In [ ]:
# Add row metadata

silver_metadata = de_dup.with_columns(
				pl.lit((datetime.now().strftime("%Y-%m-%dT%H:%M:%S"))).alias("silver_date_processed"),
				pl.lit("0001").alias("run_id"),
				pl.lit("pv_00001").alias("pipeline_version"),
)


# Write a silver cleaned source
silver_metadata.write_parquet("data/silver_pre_nf.parquet")

In [ ]:
# Resolution rules

In [1]:
import duckdb
# Write to the 3nf model
con = duckdb.connect("data/silver_3nf.duckdb")

con.execute("""
    CREATE SEQUENCE seq_stadium_id START 1;
    CREATE TABLE stadium (
        stadium_id INTEGER PRIMARY KEY DEFAULT nextval('seq_stadium_id'),
        name        VARCHAR NOT NULL,
        address     VARCHAR ,
        city		VARCHAR
        
    );
	
    CREATE SEQUENCE seq_country_id START 1;   
    CREATE TABLE country (
        country_id 	INTEGER PRIMARY KEY DEFAULT nextval('seq_country_id'),
        country    	STRING UNIQUE
    );
            
    CREATE SEQUENCE seq_team_id START 1;
	CREATE TABLE team (
        team_id INTEGER PRIMARY KEY DEFAULT nextval('seq_team_id'),
        name    STRING NOT NULL
    );

    CREATE SEQUENCE seq_match_id START 1;
	CREATE TABLE match (
        match_id    INTEGER PRIMARY KEY DEFAULT nextval('seq_match_id'),
        match_date 	DATE NOT NULL, 
        home_team 	INTEGER NOT NULL,
        away_team  	INTEGER NOT NULL,
        home_score 	INT NOT NULL DEFAULT 0,
        away_score	INT NOT NULL DEFAULT 0,
        FOREIGN KEY (home_team) REFERENCES team(team_id),
        FOREIGN KEY (away_team) REFERENCES team(team_id),
        FOREIGN KEY (away_team) REFERENCES team(team_id)
    );
                   
""")
# Add business keys (surrgate keys get added by the table)
#matches, teams, venues, competitions